In [ ]:
# Matched Training-Size Experiment

# This experiment tests whether the observed Russian versus non-Russian performance
# difference can be explained simply by training-set size. Non-Russian samples
# are therefore reduced to the size of the Russian development set.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    StratifiedShuffleSplit
)

from src.config import (
    TEXT_COL,
    LABEL_COL,
    GROUP_COL,
    RESULTS_MAIN_DIR,
    RESULTS_ROBUSTNESS_DIR,
)

from src.data import (
    load_preprocessed_data
)

from src.models import (
    fit_predict_classical
)

from src.evaluation import (
    evaluate_predictions
)


ru_df, nonru_df = (
    load_preprocessed_data()
)


ru_dev = (
    ru_df[
        ru_df["split"]
        .isin(["train", "val"])
    ]
    .copy()
)

ru_test = (
    ru_df[
        ru_df["split"] == "test"
    ]
    .copy()
)

nonru_dev = (
    nonru_df[
        nonru_df["split"]
        .isin(["train", "val"])
    ]
    .copy()
)


MATCHED_SIZE = len(ru_dev)

MATCHED_SEEDS = [
    101, 202, 303, 404, 505,
    606, 707, 808, 909, 1010
]

In [ ]:
## Construct Stratified Non-Russian Samples

# A stratified sampling procedure is defined to draw non-Russian training sets
# with exactly the same number of records as the Russian development set
# while approximately preserving the non-Russian CWE distribution.

In [2]:
def create_matched_sample(
    df,
    seed
):

    splitter = (
        StratifiedShuffleSplit(
            n_splits=1,
            train_size=MATCHED_SIZE,
            random_state=seed
        )
    )

    idx, _ = next(
        splitter.split(
            np.zeros(len(df)),
            df[LABEL_COL]
        )
    )

    return (
        df.iloc[idx]
        .copy()
        .reset_index(drop=True)
    )

In [ ]:
## Evaluate Matched Non-Russian Training Sets

# Ten independently sampled non-Russian datasets are created and evaluated using the selected
# classical models. Leakage isolation and complete 30-class coverage are verified for every run.

In [3]:
models = {
    "Logistic Regression":
        "word",

    "Linear SVM":
        "char",
}


results = []


for seed in MATCHED_SEEDS:

    sample = create_matched_sample(
        nonru_dev,
        seed
    )

    assert len(sample) == MATCHED_SIZE
    assert sample[LABEL_COL].nunique() == 30

    assert not (
        set(sample[GROUP_COL])
        &
        set(ru_test[GROUP_COL])
    )


    for model_name, representation in (
        models.items()
    ):

        pred, n_features, runtime = (
            fit_predict_classical(
                sample,
                ru_test,
                model_name,
                representation,
                TEXT_COL,
                LABEL_COL
            )
        )

        metrics, _ = (
            evaluate_predictions(
                ru_test[LABEL_COL],
                pred
            )
        )

        results.append({
            "seed":
                seed,

            "model":
                model_name,

            "train_records":
                len(sample),

            **metrics,

            "n_features":
                n_features,

            "runtime_seconds":
                runtime,
        })


matched_results = pd.DataFrame(
    results
)

In [ ]:
## Summarise Repeated Matched-Size Runs

# Macro-F1 is aggregated across the ten random samples to report the mean,
# standard deviation, minimum, and maximum performance for each model.

In [4]:
matched_summary = (
    matched_results
    .groupby("model")
    .agg(
        macro_f1_mean=(
            "macro_f1",
            "mean"
        ),
        macro_f1_sd=(
            "macro_f1",
            "std"
        ),
        macro_f1_min=(
            "macro_f1",
            "min"
        ),
        macro_f1_max=(
            "macro_f1",
            "max"
        )
    )
    .reset_index()
)


matched_summary

,model,macro_f1_mean,macro_f1_sd,macro_f1_min,macro_f1_max
0,Linear SVM,0.832018,0.003884,0.827007,0.837890
1,Logistic Regression,0.800871,0.005600,0.790222,0.806632


In [ ]:
## Compare Matched and Full Training Conditions

# Matched-size non-Russian performance is compared with the original Russian
# and full non-Russian training results. This separates the effect of training-domain
# composition from the advantage of having more training observations.

In [5]:
main_results = pd.read_csv(
    RESULTS_MAIN_DIR
    / "classical_cross_domain_all_results.csv"
)


comparison = []


for model_name in models:

    current = (
        main_results[
            main_results["model"]
            == model_name
        ]
        .set_index(
            "training_condition"
        )
    )

    matched = (
        matched_summary[
            matched_summary["model"]
            == model_name
        ]
        .iloc[0]
    )


    comparison.append({

        "model":
            model_name,

        "RU_macro_f1":
            current.loc[
                "RU",
                "macro_f1"
            ],

        "NonRU_matched_macro_f1":
            matched[
                "macro_f1_mean"
            ],

        "NonRU_matched_sd":
            matched[
                "macro_f1_sd"
            ],

        "matched_domain_gap":
            (
                current.loc[
                    "RU",
                    "macro_f1"
                ]
                -
                matched[
                    "macro_f1_mean"
                ]
            ),

        "NonRU_full_macro_f1":
            current.loc[
                "Non-RU",
                "macro_f1"
            ],
    })


matched_comparison = (
    pd.DataFrame(comparison)
)

matched_comparison

,model,RU_macro_f1,NonRU_matched_macro_f1,NonRU_matched_sd,matched_domain_gap,NonRU_full_macro_f1
0,Logistic Regression,0.866775,0.800871,0.005600,0.065904,0.826679
1,Linear SVM,0.884289,0.832018,0.003884,0.052271,0.835386


In [ ]:
## Save Matched-Size Results

# Individual runs, aggregate statistics, and the comparison
# with the main experiment are exported as robustness-analysis outputs.

In [6]:
matched_results.to_csv(
    RESULTS_ROBUSTNESS_DIR
    / "matched_size_all_runs.csv",
    index=False
)

matched_summary.to_csv(
    RESULTS_ROBUSTNESS_DIR
    / "matched_size_summary.csv",
    index=False
)

matched_comparison.to_csv(
    RESULTS_ROBUSTNESS_DIR
    / "matched_size_comparison.csv",
    index=False
)